# OpenBridge · 05 — Validación del modelo relacional (solo lectura)

**Versión:** 0.1 · **Estado:** auditoría técnica, **no** migración ni contrato aprobado.  
**Fuentes de referencia:** notebooks `01`–`04`, especialmente `04_eda_y_contrato_ia.ipynb`, y borrador de contrato Ingesta–IA (T05).  
**Alcance:** PostgreSQL usado por el catálogo GitHub y la interfaz del módulo IA.

### Preguntas que intentamos responder

1. ¿Existen realmente las tablas `catalog_*`, la vista `v_repos_for_ai` y las tablas `ai_*`?
2. ¿Qué tipos, nulabilidades, claves, relaciones, índices y permisos existen actualmente?
3. ¿Cumple el esquema los puntos verificables del contrato T05?
4. ¿Hay indicios de inconsistencia en identificadores, READMEs, forks, `homepage` o `content_hash`?

> **Seguridad:** las consultas de esta notebook son `SELECT`. La conexión solicita `default_transaction_read_only=on` y un tiempo máximo por sentencia. **No incluye** `CREATE`, `ALTER`, `DROP`, `UPDATE`, `DELETE`, `INSERT`, ni escrituras locales automáticas. No ejecuta extracción de GitHub.
>
> **Límite:** resultados actuales únicamente cuando tú ejecutes esta notebook en tu entorno. Los resultados guardados en notebook 04 no certifican el estado de PostgreSQL hoy.

## 0. Preparación segura

Usamos los **mismos nombres de variables de entorno** que la notebook 04:

- `DB_USER` (**obligatoria**)
- `DB_PASSWORD` (**obligatoria**)
- `DB_HOST` (por defecto `localhost`)
- `DB_PORT` (por defecto `5432`)
- `DB_NAME` (por defecto `openbridge_db`)
- `DB_SCHEMA` (opcional, por defecto `public`)

Se necesita un entorno Python con `sqlalchemy`, `psycopg` v3, `pandas`, `python-dotenv` e `IPython` instalados **previamente**. Esta notebook **no instala dependencias**.

Si utilizas `.env`, debe estar en el proyecto o en una ruta padre desde la que pueda detectarse. **Nunca pegues contraseñas en celdas ni compartas un `.env`**. Recomendado: ejecutar las consultas con un usuario PostgreSQL que solo tenga permisos de lectura.

**Orden:** ejecuta las celdas secuencialmente. Si falla la conexión o el modo de solo lectura, detente y comparte **solo el mensaje de error sin credenciales**.

In [ ]:
import os
import re
import pandas as pd
from dotenv import load_dotenv, find_dotenv
from IPython.display import display
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

ruta_env = find_dotenv(usecwd=True)
if ruta_env:
    load_dotenv(ruta_env, override=False)

DB_SCHEMA = os.getenv('DB_SCHEMA', 'public').strip()
DB_HOST = os.getenv('DB_HOST', 'localhost')
DB_PORT = os.getenv('DB_PORT', '5432')
DB_NAME = os.getenv('DB_NAME', 'openbridge_db')
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')

if not DB_USER or not DB_PASSWORD:
    raise RuntimeError(
        'Faltan DB_USER o DB_PASSWORD. Configúralas en el entorno o en un .env local; '
        'no las escribas en la notebook.'
    )
if not DB_SCHEMA:
    raise ValueError('DB_SCHEMA no puede estar vacío.')

# URL.create codifica correctamente contraseñas con @, :, /, %, etc.
url_db = URL.create(
    'postgresql+psycopg',
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=int(DB_PORT),
    database=DB_NAME,
)

# Configuración de sesión (sin alterar datos ni esquema).
engine = create_engine(
    url_db,
    pool_pre_ping=True,
    connect_args={
        'options': '-c default_transaction_read_only=on -c statement_timeout=15000'
    },
)

with engine.connect() as conn:
    estado = conn.execute(text('''SELECT
        current_database() AS base_datos,
        current_schema() AS esquema_por_defecto,
        current_user AS rol_conectado,
        current_setting('transaction_read_only') AS solo_lectura,
        current_setting('statement_timeout') AS limite_sentencia
    ''')).mappings().one()

if estado['solo_lectura'] != 'on':
    raise RuntimeError('La sesión no es de solo lectura. No ejecutar otras celdas.')

display(pd.DataFrame([dict(estado)]))
print('Conexión verificada en modo SOLO LECTURA. Esquema objetivo:', DB_SCHEMA)

## 1. Inventario de objetos del contrato

Distinguimos objetos **observados en PostgreSQL** de objetos **descritos en T05**. La ausencia de una tabla `ai_*` no significa, por sí sola, que exista un error: el módulo de IA puede estar todavía en diseño.

In [ ]:
OBJETOS = {
    'catalog_repositories': 'Tabla de ingesta · esperada',
    'catalog_readmes': 'Tabla de ingesta · esperada',
    'v_repos_for_ai': 'Vista de contrato · esperada',
    'ai_categories': 'IA · propuesta en T05',
    'ai_repo_categories': 'IA · propuesta en T05',
    'ai_integration_ideas': 'IA · propuesta en T05',
    'ai_embeddings': 'IA · propuesta en T05',
}
TABLAS_CATALOGO = ('catalog_repositories', 'catalog_readmes')
TABLAS_IA = ('ai_categories', 'ai_repo_categories', 'ai_integration_ideas', 'ai_embeddings')

with engine.connect() as conn:
    inventario_bruto = pd.read_sql_query(text('''
        SELECT table_name, table_type
        FROM information_schema.tables
        WHERE table_schema = :esquema
        ORDER BY table_name
    '''), conn, params={'esquema': DB_SCHEMA})

objetos_reales = dict(zip(inventario_bruto['table_name'], inventario_bruto['table_type']))
df_inventario = pd.DataFrame([
    {
        'objeto': nombre,
        'responsabilidad': descripcion,
        'existe_en_esquema': nombre in objetos_reales,
        'tipo_observado': objetos_reales.get(nombre, 'NO ENCONTRADO'),
    }
    for nombre, descripcion in OBJETOS.items()
])
display(df_inventario)

## 2. Diccionario real de columnas

Recuperamos **metadata**; no leemos valores de columnas como `readme_text`. Una definición observada puede diferir del diseño mostrado en el HTML de trabajo: ese desacuerdo es precisamente lo que debemos documentar.

In [ ]:
with engine.connect() as conn:
    columnas_todas = pd.read_sql_query(text('''
        SELECT table_name, ordinal_position, column_name, data_type,
               udt_name, is_nullable, character_maximum_length,
               column_default
        FROM information_schema.columns
        WHERE table_schema = :esquema
        ORDER BY table_name, ordinal_position
    '''), conn, params={'esquema': DB_SCHEMA})

columnas = columnas_todas[columnas_todas['table_name'].isin(OBJETOS)].copy()
campos_por_objeto = {
    nombre: set(columnas.loc[columnas['table_name'].eq(nombre), 'column_name'])
    for nombre in OBJETOS
}
display(columnas.reset_index(drop=True))

## 3. Claves, restricciones y relaciones comprobables

Consultamos `pg_catalog.pg_constraint` para identificar las **restricciones reales**. No inferimos relaciones a partir de nombres parecidos: solo consideramos respaldada una FK cuando PostgreSQL la declara.

In [ ]:
with engine.connect() as conn:
    restricciones_brutas = pd.read_sql_query(text('''
        SELECT c.relname AS tabla,
               x.conname AS nombre_restriccion,
               x.contype AS tipo_codigo,
               CASE x.contype
                   WHEN 'p' THEN 'PRIMARY KEY'
                   WHEN 'f' THEN 'FOREIGN KEY'
                   WHEN 'u' THEN 'UNIQUE'
                   WHEN 'c' THEN 'CHECK'
                   ELSE x.contype::text
               END AS tipo,
               ref.relname AS tabla_referenciada,
               pg_get_constraintdef(x.oid, true) AS definicion
        FROM pg_catalog.pg_constraint x
        JOIN pg_catalog.pg_class c ON c.oid = x.conrelid
        JOIN pg_catalog.pg_namespace n ON n.oid = c.relnamespace
        LEFT JOIN pg_catalog.pg_class ref ON ref.oid = x.confrelid
        WHERE n.nspname = :esquema
        ORDER BY c.relname, tipo, x.conname
    '''), conn, params={'esquema': DB_SCHEMA})

restricciones = restricciones_brutas[restricciones_brutas['tabla'].isin(OBJETOS)].copy()
display(restricciones.reset_index(drop=True))

## 4. Índices y permisos de la cuenta conectada

- Los índices ayudan a validar decisiones de rendimiento (por ejemplo, `topics` / GIN), pero **no** demuestran calidad de datos.
- Los permisos que vemos corresponden **solo a la cuenta actual**. No prueban que todos los roles estén correctamente aislados; habría que auditar los roles de la ingesta y la IA por separado.

In [ ]:
with engine.connect() as conn:
    indices_brutos = pd.read_sql_query(text('''
        SELECT tablename AS tabla, indexname AS indice, indexdef AS definicion
        FROM pg_catalog.pg_indexes
        WHERE schemaname = :esquema
        ORDER BY tablename, indexname
    '''), conn, params={'esquema': DB_SCHEMA})
    permisos_brutos = pd.read_sql_query(text('''
        SELECT c.relname AS objeto,
               has_table_privilege(current_user, c.oid, 'SELECT') AS puede_select,
               has_table_privilege(current_user, c.oid, 'INSERT') AS puede_insert,
               has_table_privilege(current_user, c.oid, 'UPDATE') AS puede_update,
               has_table_privilege(current_user, c.oid, 'DELETE') AS puede_delete
        FROM pg_catalog.pg_class c
        JOIN pg_catalog.pg_namespace n ON n.oid = c.relnamespace
        WHERE n.nspname = :esquema AND c.relkind IN ('r','p','v','m')
        ORDER BY c.relname
    '''), conn, params={'esquema': DB_SCHEMA})

indices = indices_brutos[indices_brutos['tabla'].isin(OBJETOS)].copy()
permisos = permisos_brutos[permisos_brutos['objeto'].isin(OBJETOS)].copy()
print('Índices observados:')
display(indices.reset_index(drop=True))
print('Privilegios del rol con el que está conectada esta notebook:')
display(permisos.reset_index(drop=True))

## 5. Inspección de la vista `v_repos_for_ai`

La notebook 04 definía esta vista mediante un `LEFT JOIN` sobre `github_id`. Aquí comprobamos la **definición actualmente registrada en PostgreSQL**. No ejecutamos ni reemplazamos la vista.

In [ ]:
if objetos_reales.get('v_repos_for_ai') == 'VIEW':
    with engine.connect() as conn:
        df_definicion_vista = pd.read_sql_query(text('''
            SELECT viewname, definition
            FROM pg_catalog.pg_views
            WHERE schemaname = :esquema AND viewname = :nombre
        '''), conn, params={'esquema': DB_SCHEMA, 'nombre': 'v_repos_for_ai'})
    if df_definicion_vista.empty:
        print('Vista detectada, pero definición no accesible con este rol.')
    else:
        print(df_definicion_vista.loc[0, 'definition'])
else:
    print('No se encontró v_repos_for_ai como VIEW ordinaria en el esquema elegido.')

## 6. Revisión estructural del contrato T05

**Interpretación:** `CUMPLE` = condición estructural respaldada por metadata; `NO CUMPLE` = diferencia concreta; `PENDIENTE` = objeto aún no existe o no es verificable en esta auditoría. Un resultado `CUMPLE` **no** equivale a aprobar todas las reglas de negocio.

In [ ]:
def campo(tabla, nombre):
    return nombre in campos_por_objeto.get(tabla, set())

def filas_restriccion(tabla, tipo):
    return restricciones.loc[
        restricciones['tabla'].eq(tabla) & restricciones['tipo_codigo'].eq(tipo)
    ]

revision = []
def agregar(criterio, estado, evidencia, alcance='Estructura'):
    revision.append({'criterio': criterio, 'resultado': estado,
                     'evidencia_o_limitacion': evidencia, 'alcance': alcance})

for tabla in TABLAS_CATALOGO:
    agregar(f'Existe {tabla}',
            'CUMPLE' if objetos_reales.get(tabla) == 'BASE TABLE' else 'NO CUMPLE',
            'Objeto observado en information_schema.tables' if tabla in objetos_reales
            else 'No se encontró como tabla base')

if objetos_reales.get('catalog_repositories') == 'BASE TABLE':
    es_bigint = bool(((columnas['table_name'] == 'catalog_repositories') &
                      (columnas['column_name'] == 'github_id') &
                      (columnas['udt_name'] == 'int8')).any())
    pk = filas_restriccion('catalog_repositories', 'p')
    pk_github = any(re.search(r'\\bgithub_id\\b', str(x)) for x in pk['definicion'])
    agregar('github_id BIGINT en catalog_repositories', 'CUMPLE' if es_bigint else 'NO CUMPLE',
            'Tipo PostgreSQL int8 = BIGINT')
    agregar('PRIMARY KEY de catalog_repositories sobre github_id',
            'CUMPLE' if pk_github else 'NO CUMPLE',
            'Revisar la definición de PRIMARY KEY observada')
    for nombre in ('homepage', 'content_hash', 'pushed_at', 'license_spdx_id'):
        agregar(f'Columna catalog_repositories.{nombre}',
                'CUMPLE' if campo('catalog_repositories', nombre) else 'NO CUMPLE',
                'Presencia en information_schema.columns; no valida semántica')
else:
    agregar('Estructura catalog_repositories', 'PENDIENTE', 'No se encontró tabla base')

if objetos_reales.get('catalog_readmes') == 'BASE TABLE':
    pk_readme = filas_restriccion('catalog_readmes', 'p')
    fk_readme = filas_restriccion('catalog_readmes', 'f')
    pk_ok = any(re.search(r'\\bgithub_id\\b', str(x)) for x in pk_readme['definicion'])
    fk_ok = any(
        (str(r['tabla_referenciada']) == 'catalog_repositories') and
        bool(re.search(r'\\bgithub_id\\b', str(r['definicion'])))
        for _, r in fk_readme.iterrows()
    )
    agregar('catalog_readmes: PRIMARY KEY sobre github_id',
            'CUMPLE' if pk_ok else 'NO CUMPLE', 'Restricción PRIMARY KEY declarada')
    agregar('catalog_readmes: FK hacia catalog_repositories',
            'CUMPLE' if fk_ok else 'NO CUMPLE', 'Restricción FOREIGN KEY declarada')
    agregar('catalog_readmes.readme_text',
            'CUMPLE' if campo('catalog_readmes', 'readme_text') else 'NO CUMPLE',
            'Presencia de columna; contenido no inspeccionado')
else:
    agregar('Estructura catalog_readmes', 'PENDIENTE', 'No se encontró tabla base')

agregar('Vista v_repos_for_ai',
        'CUMPLE' if objetos_reales.get('v_repos_for_ai') == 'VIEW' else 'NO CUMPLE',
        'Presencia como VIEW (no implica que sus permisos o contenido sean correctos)')

if objetos_reales.get('v_repos_for_ai') == 'VIEW':
    for nombre in ('github_id', 'full_name', 'readme_text', 'content_hash', 'license_spdx_id', 'archived'):
        agregar(f'Campo de vista v_repos_for_ai.{nombre}',
                'CUMPLE' if campo('v_repos_for_ai', nombre) else 'NO CUMPLE',
                'Presencia en information_schema.columns; no verifica filas')

for tabla in TABLAS_IA:
    agregar(f'{tabla} (T05)',
            'ENCONTRADA' if objetos_reales.get(tabla) == 'BASE TABLE' else 'PENDIENTE',
            'Existencia observada; DDL exacto y responsabilidad requieren revisión',
            'Contrato propuesto')

df_revision = pd.DataFrame(revision)
display(df_revision)

## 7. Calidad de datos: mediciones agregadas, sin exponer READMEs

Estas consultas cuentan registros y anomalías. **No exportan textos ni identificadores concretos**. Una ausencia de README o un `content_hash` nulo puede tener explicación funcional: la interpretación corresponde al equipo.

Cada consulta está condicionada a la existencia de las tablas y columnas. El esquema y los nombres de tabla se citan como identificadores SQL escapados, en lugar de interpolar texto del usuario sin validar.

> Precaución: los `COUNT(*)` pueden examinar muchas filas. Aunque hay un `statement_timeout` de 15 segundos, en una base de datos grande conviene revisar el coste antes de ejecutar.

In [ ]:
# El esquema viene del entorno; SQLAlchemy escapa su nombre como identificador.
preparador = engine.dialect.identifier_preparer

def tabla_sql(nombre):
    if nombre not in OBJETOS:
        raise ValueError('Objeto fuera de lista blanca')
    return f'{preparador.quote_schema(DB_SCHEMA)}.{preparador.quote(nombre)}'

def disponible(tabla, *campos):
    return objetos_reales.get(tabla) == 'BASE TABLE' and all(
        campo(tabla, columna) for columna in campos
    )

metricas = []
def medir(etiqueta, consulta, interpretacion):
    if not consulta.lstrip().upper().startswith('SELECT'):
        raise ValueError('Esta notebook solo permite consultas SELECT para métricas')
    try:
        with engine.connect() as conn:
            valor = conn.execute(text(consulta)).scalar_one()
        metricas.append({'indicador': etiqueta, 'valor': int(valor or 0),
                         'interpretacion': interpretacion, 'estado': 'MEDIDO'})
    except Exception as exc:
        # No incluir texto SQL, DSN ni credenciales en el resultado.
        metricas.append({'indicador': etiqueta, 'valor': None,
                         'interpretacion': interpretacion,
                         'estado': 'NO MEDIDO: ' + type(exc).__name__})

def omitir(etiqueta, razon):
    metricas.append({'indicador': etiqueta, 'valor': None,
                     'interpretacion': razon, 'estado': 'NO APLICA'})

for tabla in TABLAS_CATALOGO:
    if disponible(tabla):
        medir(f'Filas en {tabla}', f'SELECT COUNT(*) FROM {tabla_sql(tabla)}',
              'Inventario; no es una métrica de calidad por sí sola')
    else:
        omitir(f'Filas en {tabla}', 'Tabla no encontrada')

if disponible('catalog_repositories', 'github_id'):
    medir('Repositorios con github_id nulo',
          f'SELECT COUNT(*) FROM {tabla_sql("catalog_repositories")} WHERE github_id IS NULL',
          'Se espera 0 para un identificador obligatorio')
    medir('github_id repetidos (filas adicionales)',
          f'''SELECT COALESCE(SUM(n - 1), 0) FROM
                (SELECT COUNT(*) AS n FROM {tabla_sql('catalog_repositories')}
                 GROUP BY github_id HAVING COUNT(*) > 1) t''',
          'Se espera 0; una PRIMARY KEY también debe prevenirlos')
else:
    omitir('Calidad de github_id', 'No se encontró tabla/columna')

if disponible('catalog_readmes', 'github_id') and disponible('catalog_repositories', 'github_id'):
    medir('READMEs sin repositorio (huérfanos)',
          f'''SELECT COUNT(*) FROM {tabla_sql('catalog_readmes')} m
               LEFT JOIN {tabla_sql('catalog_repositories')} r ON m.github_id = r.github_id
               WHERE r.github_id IS NULL''',
          'Se espera 0 si se aplica correctamente la FK')
    medir('Repositorios sin fila de README',
          f'''SELECT COUNT(*) FROM {tabla_sql('catalog_repositories')} r
               LEFT JOIN {tabla_sql('catalog_readmes')} m ON m.github_id = r.github_id
               WHERE m.github_id IS NULL''',
          'Puede ser válido cuando no hay README; investigar procedencia')
else:
    omitir('Integridad entre repositorios y READMEs', 'Faltan tablas o columnas')

if disponible('catalog_repositories', 'homepage'):
    medir('homepage vacío o con espacios, no NULL',
          f'''SELECT COUNT(*) FROM {tabla_sql('catalog_repositories')}
              WHERE homepage IS NOT NULL AND BTRIM(homepage::text) = '' ''',
          'El contrato T05 pide normalizar estos valores a NULL')
else:
    omitir('Saneamiento homepage', 'Falta la columna homepage')

if disponible('catalog_repositories', 'fork'):
    medir('Forks presentes en el catálogo',
          f'SELECT COUNT(*) FROM {tabla_sql("catalog_repositories")} WHERE fork IS TRUE',
          'T05 propone excluir forks de la ingesta')
else:
    omitir('Exclusión de forks', 'Sin columna fork no es verificable con esta consulta')

if disponible('catalog_repositories', 'content_hash'):
    medir('content_hash ausentes',
          f'SELECT COUNT(*) FROM {tabla_sql("catalog_repositories")} WHERE content_hash IS NULL',
          'Investigar si hay registros sin README o pendientes de procesamiento')
    medir('content_hash con formato distinto a 64 caracteres hexadecimales',
          f'''SELECT COUNT(*) FROM {tabla_sql('catalog_repositories')}
              WHERE content_hash IS NOT NULL
                AND content_hash::text !~ '^[0-9a-fA-F]{{64}}$' ''',
          'Solo valida formato SHA-256; NO demuestra que el cálculo sea correcto')
else:
    omitir('Formato de content_hash', 'Falta la columna content_hash')

if disponible('catalog_readmes', 'readme_text'):
    medir('READMEs NULL o vacíos',
          f'''SELECT COUNT(*) FROM {tabla_sql('catalog_readmes')}
              WHERE readme_text IS NULL OR BTRIM(readme_text) = '' ''',
          'Muestra de cobertura; no necesariamente error de ingesta')
else:
    omitir('Cobertura de READMEs', 'Falta la columna readme_text')

df_metricas = pd.DataFrame(metricas)
display(df_metricas)

## 8. Informe de contraste, alcance y decisiones abiertas

Completa este resumen **después de ejecutar** todas las celdas en tu equipo. El análisis aquí no aprueba cambios.

| Tema | Qué se puede determinar aquí | Qué NO se puede determinar todavía |
|---|---|---|
| Identidad | PK y tipo SQL de `github_id` | Identificador canónico entre GitHub, GitLab, PyPI, npm, etc. |
| Contrato con IA | Vista, columnas y FKs registradas | Que el consumidor de IA procese correctamente la vista |
| Hash | Presencia y formato hexadecimal | Fidelidad a `SHA256(readme_text + pushed_at)` y frecuencia de reproceso |
| Integridad | Conteos agregados y huérfanos | Exactitud, actualidad y procedencia de cada README |
| Seguridad | Privilegios del rol actual | Separación real de roles entre ingesta e IA |
| Escalabilidad | Índices existentes | Rendimiento con cargas y volúmenes de producción |

### Criterios de revisión conjunta

- [ ] Confirmar que `catalog_repositories.github_id` es `BIGINT PRIMARY KEY`.
- [ ] Confirmar PK/FK sobre `catalog_readmes.github_id`.
- [ ] Comparar la definición de `v_repos_for_ai` con T05 (columnas, `LEFT JOIN`, filtrado).
- [ ] Clasificar explícitamente el estado de las cuatro tablas `ai_*`.
- [ ] Revisar métricas de nulos, forks, `homepage` y hashes sin interpretar los resultados automáticamente como fallos.
- [ ] Abrir una decisión de diseño sobre la identidad entre ecosistemas.
- [ ] Revisar el HTML del modelo relacional con el equipo de ingesta y el equipo IA.

**Siguiente paso sugerido:** compartir los resultados de las celdas **sin credenciales ni valores de README**; juntos documentaremos diferencias y un plan de corrección. Ninguna corrección de esquema debe ejecutarse dentro de esta notebook de auditoría.